## **Imports**

In [2]:
!pip install -U transformers accelerate torch pillow
!pip install hf_xet

  Using cached torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached torch-2.11.0-cp312-cp312-win_amd64.whl (114.6 MB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)

  Attempting uninstall: sympy

    Found existing installation: sympy 1.13.1

   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
    Uninstalling sympy-1.13.1:
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ---------------------------------------- 0/3 [sympy]
   ----------


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from millionaire_client import AuthenticationError, MillionaireClient
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor
from src.guesser.guesser import Guesser

## **Auth**

In [5]:
API_URL = "http://131.175.15.22:51111/"
USERNAME = "MTKY"
PASSWORD = "Marcelo2504#"
TOKEN =  "YOUR_HF_TOKEN_HERE"

In [6]:
client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


Welcome, MTKY! (Role: student)

=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


## **Model Definition**

In [7]:
MODEL_ID = "google/gemma-4-e2b-it"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=TOKEN,
    device_map="cpu"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## **Quiz**

In [ ]:
# Prompt

system_instructions = {"role": "system", 
    "content":
    """
    You are an expert data extraction and Q&A assistant. Your task is to read the provided context and answer the multiple-choice question. 
    CRITICAL RULE: You must reply ONLY with the exact letter of the correct option (A, B, C, or D). Do not provide any explanations, introductory phrases, or punctuation other than the single uppercase letter.
    Question: Which of the following best describes Sicily?
    A) A small lake in northern Europe
    B) A landlocked country in Asia
    C) The largest island in the Mediterranean Sea
    D) A mountain range in France
    Answer: C

    Question: {user_question}
    A) {option_a}
    B) {option_b}
    C) {option_c}
    D) {option_d}
    Answer:
    """
}
messages = [
    system_instructions,
    {"role": "user", "content": "Write a short joke about saving RAM."},
]

# Process input
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=False
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


{'role': 'assistant', 'content': 'B'}

In [ ]:
from src.context_db.context_db import contextDB

client = contextDB()

Initializing ChromaDB at: c:\Users\marce\Desktop\Code\NLP_Assignment\Marcelo\src\context_db


In [3]:
client.get_all_collections()
client.create_collection("test")

[]


Collection(name=test)

In [ ]:
client.add_document_to_collection("test", ["Marcelo tem 20 anos", "A casa é verde", "A internet é tóxica"],None)

ValueError: Expected metadata to be a non-empty dict, got 0 metadata attributes in upsert.